# MATCHA-Bench on Kaggle

GPU stage of the pipeline: lexicon-guided DAPT and fine-tuning.

**Before running**
1. Accelerator: **GPU T4 x2** (Settings → phone verification unlocks GPUs).
2. Attach the private dataset with `data/` and `lexicons/` prepared locally (see `docs/KAGGLE.md`).
3. Attach or clone the code.
4. For long jobs use *Save Version → Save & Run All* (up to 12 h headless).

In [ ]:
# ---- paths: edit these to match your attached datasets ----
DATA_INPUT = "/kaggle/input/matcha-bench-data"     # contains data/ and lexicons/
CODE_INPUT = "/kaggle/input/matcha-bench-code"     # repo folder, or set to None to git clone
REPO = "/kaggle/working/matcha-bench"
GIT_URL = None                                      # e.g. "https://github.com/<org>/matcha-bench.git"
VARIANT = "random"                                  # random | depression | multidisorder

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, shutil, subprocess

if GIT_URL:
    subprocess.run(["git", "clone", "--depth", "1", GIT_URL, REPO], check=True)
elif not os.path.exists(REPO):
    shutil.copytree(CODE_INPUT, REPO)

# data/ and lexicons/ are read from the input dataset; outputs/ stays in /kaggle/working
for name in ("data", "lexicons"):
    target = os.path.join(REPO, name)
    if os.path.islink(target) or not os.path.exists(target):
        if os.path.islink(target):
            os.unlink(target)
        os.symlink(os.path.join(DATA_INPUT, name), target)
    else:
        shutil.copytree(os.path.join(DATA_INPUT, name), target, dirs_exist_ok=True)

os.chdir(REPO)
!pip install -q jieba
!ls data lexicons

## 1. Smoke test (2 minutes)
A tiny randomly-initialised BERT for 20 steps, to check that the whole stack runs.

In [ ]:
!python scripts/run_dapt.py --config configs/dapt/smoke.yaml

## 2. Domain-adaptive pre-training
Run one variant per session (about 1.5–3 h on T4 x2). It resumes automatically if
`outputs/dapt/<variant>` from a previous session is copied back in.

In [ ]:
!python scripts/run_dapt.py --config configs/dapt/{VARIANT}.yaml
!cat outputs/dapt/{VARIANT}/final/run_info.json | head -40

## 3. Fine-tuning
Already-finished `(task, model, seed)` runs are skipped, so this cell can be re-run after a timeout.

In [ ]:
ACL_TASKS = "ewect_usual ewect_epidemic waimai cognitive suicide"
MATCHA_TASKS = ("matcha_ip_post matcha_disorder_post matcha_disorder_post_masked "
                "matcha_user_binary matcha_user_binary_masked matcha_user_multiclass matcha_user_multiclass_masked")
MODELS = f"bert-wwm-ext chinese-mentalbert matcha-{VARIANT}"

!python scripts/run_finetune.py --tasks {ACL_TASKS} --models {MODELS}

In [ ]:
!python scripts/run_finetune.py --tasks {MATCHA_TASKS} --models {MODELS}

## 4. Results

In [ ]:
!python scripts/summarize_results.py
!python scripts/summarize_results.py --compare matcha-{VARIANT} bert-wwm-ext

In [ ]:
# Keep outputs small: drop intermediate DAPT checkpoints once the final model exists.
import glob
for ckpt in glob.glob(f"outputs/dapt/*/checkpoint-*"):
    if os.path.exists(os.path.join(os.path.dirname(ckpt), "final")):
        shutil.rmtree(ckpt)
!du -sh outputs/*